In [ ]:
pip install torch torchvision timm thop pytorch-msssim Pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import csv
import time
import math
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from pytorch_msssim import ssim as ssim_func

In [ ]:
!gdown https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T

Downloading...
From (original): https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T
From (redirected): https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T&confirm=t&uuid=f3ae972e-c4f5-4bb2-bcbb-3c36414dea5b
To: /content/Final_Modeling_Dataset.zip
100% 163M/163M [00:02<00:00, 77.1MB/s]


In [ ]:
!unzip Final_Modeling_Dataset.zip -d /content/Final_Modeling_Dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/Final_Modeling_Dataset/train/blur/1074_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1315_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1211_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/535_very_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/792_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1898_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1890_low.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1608_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1460_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1225_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/105_low.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/321_very_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1629_very_high.jpg 

In [ ]:
DATASET_DIR   = "/content/Final_Modeling_Dataset"
RESULTS_DIR   = "/content/drive/MyDrive/Vit_U_Net_Result"
CHECKPOINT_DIR = os.path.join(RESULTS_DIR, "checkpoints")
OUTPUT_DIR    = os.path.join(RESULTS_DIR, "deblurred_outputs")

In [ ]:
IMG_SIZE      = 256
BATCH_SIZE    = 8
EPOCHS        = 30
LR            = 1e-4
WEIGHT_DECAY  = 1e-2
PATIENCE      = 10       # early stopping patience
SAVE_EVERY    = 5        # save checkpoint every N epochs

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,     exist_ok=True)

print(f"Device        : {DEVICE}")
print(f"Dataset       : {DATASET_DIR}")
print(f"Results saved : {RESULTS_DIR}\n")

Device        : cuda
Dataset       : /content/Final_Modeling_Dataset
Results saved : /content/drive/MyDrive/Vit_U_Net_Result



In [ ]:
class DeblurDataset(Dataset):
    def __init__(self, split, transform=None):
        self.sharp_dir  = os.path.join(DATASET_DIR, split, "sharp")
        self.blur_dir   = os.path.join(DATASET_DIR, split, "blur")
        self.transform  = transform
        self.pairs      = []

        blur_files = sorted([
            f for f in os.listdir(self.blur_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        for blur_file in blur_files:
            stem = os.path.splitext(blur_file)[0]
            for suffix in ["_very_high", "_high", "_medium", "_low"]:
              if stem.endswith(suffix):
                stem = stem[:-len(suffix)]
                break
            ext        = os.path.splitext(blur_file)[1]
            sharp_file = f"{stem}{ext}"
            sharp_path = os.path.join(self.sharp_dir, sharp_file)

            if os.path.exists(sharp_path):
                self.pairs.append((
                    os.path.join(self.blur_dir, blur_file),
                    sharp_path,
                    blur_file
                ))

        print(f"[{split}] Loaded {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        blur_path, sharp_path, filename = self.pairs[idx]
        blur_img  = Image.open(blur_path).convert("RGB")
        sharp_img = Image.open(sharp_path).convert("RGB")

        if self.transform:
            blur_img  = self.transform(blur_img)
            sharp_img = self.transform(sharp_img)

        return blur_img, sharp_img, filename


transform = transforms.Compose([
    transforms.ToTensor(),                                          # 0-255 → 0.0-1.0
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # 0.0-1.0 → -1.0-1.0
])

train_dataset = DeblurDataset("train",      transform)
val_dataset   = DeblurDataset("validation", transform)
test_dataset  = DeblurDataset("test",       transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=1,          shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test samples  : {len(test_dataset)}\n")

[train] Loaded 7048 pairs
[validation] Loaded 612 pairs
[test] Loaded 624 pairs

Train batches : 881
Val batches   : 77
Test samples  : 624



In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels)
        )
    def forward(self, x):
        return self.double_conv(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(embed_dim, embed_dim * 3)
        self.proj      = nn.Linear(embed_dim, embed_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv     = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv     = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn    = (q @ k.transpose(-2, -1)) * self.scale
        attn    = attn.softmax(dim=-1)
        attn    = self.dropout(attn)
        x       = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x       = self.proj(x)
        return x

In [ ]:
class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViTUNetDeblur(nn.Module):
    def __init__(self, img_size=256, in_channels=3, out_channels=3,
                 features=[64, 128, 256, 512], embed_dim=512,
                 depth=4, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder (CNN)
        curr_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(curr_channels, feature))
            curr_channels = feature

        # Bottleneck (ViT)
        # Menghitung ukuran resolusi spasial setelah pooling berlapis
        # 256 -> 128 -> 64 -> 32 -> 16 (4 kali downsampling)
        bottleneck_size = img_size // (2 ** len(features))
        self.num_patches = bottleneck_size * bottleneck_size

        self.bottleneck_conv_in = nn.Conv2d(features[-1], embed_dim, kernel_size=1)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.transformer_norm = nn.LayerNorm(embed_dim)
        self.bottleneck_conv_out = nn.Conv2d(embed_dim, features[-1] * 2, kernel_size=1)

        # Decoder (CNN)
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2)
            )
            self.ups.append(DoubleConv(feature * 2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        skip_connections = []

        # Downward path
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        # Transformer Bottleneck
        x = self.bottleneck_conv_in(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.transformer_blocks:
            x = block(x)

        x = self.transformer_norm(x)
        x = x.transpose(1, 2).reshape(B, C, H, W)
        x = self.bottleneck_conv_out(x)

        # Upward path
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]

            if x.shape != skip_connection.shape:
                diffY = skip_connection.size()[2] - x.size()[2]
                diffX = skip_connection.size()[3] - x.size()[3]
                x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                              diffY // 2, diffY - diffY // 2])

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return torch.tanh(self.final_conv(x))

In [ ]:
class L1SSIMLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        pred_01   = (pred + 1) / 2
        target_01 = (target + 1) / 2
        l1_loss   = self.l1(pred, target)
        ssim_loss = 1 - ssim_func(pred_01, target_01, data_range=1.0, size_average=True)
        return self.alpha * l1_loss + (1 - self.alpha) * ssim_loss

In [ ]:
def compute_psnr(pred, target):
    pred   = (pred.clamp(-1, 1) + 1) / 2
    target = (target.clamp(-1, 1) + 1) / 2
    mse    = torch.mean((pred - target) ** 2)
    if mse == 0: return float("inf")
    return 20 * torch.log10(1.0 / torch.sqrt(mse)).item()

def compute_ssim(pred, target):
    pred   = (pred.clamp(-1, 1) + 1) / 2
    target = (target.clamp(-1, 1) + 1) / 2
    return ssim_func(pred, target, data_range=1.0, size_average=True).item()

def denormalize(tensor):
    tensor = (tensor.clamp(-1, 1) + 1) / 2
    tensor = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    tensor = (tensor * 255).astype(np.uint8)
    return cv2.cvtColor(tensor, cv2.COLOR_RGB2BGR)

In [ ]:
def get_model_metrics(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("=" * 50)
    print("COMPUTATIONAL METRICS")
    print("=" * 50)
    print(f"Total parameters     : {total:,}")
    print(f"Trainable parameters : {trainable:,} ({trainable/1e6:.2f}M)")

    try:
        from thop import profile
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        print(f"FLOPs                : {flops/1e9:.2f}G")
        flops_val = flops
    except ImportError:
        print("FLOPs                : N/A (Install thop: pip install thop)")
        flops_val = None
    print("=" * 50 + "\n")
    return total, trainable, flops_val

In [ ]:
def train_model(model, optimizer, criterion, scheduler=None):
    best_val_loss  = float("inf")
    patience_count = 0
    log_rows       = []
    total_params, trainable_params, flops = get_model_metrics(model)

    train_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()

        model.train()
        train_loss = 0.0
        for blur, sharp, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False):
            blur, sharp = blur.to(DEVICE), sharp.to(DEVICE)
            optimizer.zero_grad()
            output = model(blur)
            loss   = criterion(output, sharp)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = val_psnr = val_ssim = 0.0
        with torch.no_grad():
            for blur, sharp, _ in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]  ", leave=False):
                blur, sharp = blur.to(DEVICE), sharp.to(DEVICE)
                output   = model(blur)
                loss     = criterion(output, sharp)
                val_loss += loss.item()
                val_psnr += compute_psnr(output, sharp)
                val_ssim += compute_ssim(output, sharp)

        val_loss /= len(val_loader)
        val_psnr /= len(val_loader)
        val_ssim /= len(val_loader)
        epoch_time = time.time() - epoch_start

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val PSNR: {val_psnr:.2f}dB | Val SSIM: {val_ssim:.4f} | Time: {epoch_time:.1f}s")

        log_rows.append({
            "epoch": epoch, "train_loss": round(train_loss, 6), "val_loss": round(val_loss, 6),
            "val_psnr": round(val_psnr, 4), "val_ssim": round(val_ssim, 4), "epoch_time": round(epoch_time, 2)
        })

        if epoch % SAVE_EVERY == 0:
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_loss": val_loss,
            }, os.path.join(CHECKPOINT_DIR, f"hybrid_epoch_{epoch:03d}.pth"))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            torch.save(model.state_dict(), os.path.join(RESULTS_DIR, "best_model.pth"))
            print(f"  --> Best model saved (val loss: {best_val_loss:.4f})")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\nEarly stopping triggered at epoch {epoch}")
                break

        if scheduler: scheduler.step(val_loss)

    total_training_time = time.time() - train_start
    with open(os.path.join(RESULTS_DIR, "training_log.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=log_rows[0].keys())
        writer.writeheader(); writer.writerows(log_rows)

    return total_training_time, total_params, trainable_params, flops

In [ ]:
def test_model(model, return_images=True):
    print("\n" + "="*50)
    print("TESTING")
    print("="*50)

    best_path = os.path.join(RESULTS_DIR, "best_model.pth")
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()
    print(f"Loaded best model from: {best_path}")

    test_psnr = test_ssim = 0.0
    inference_times = []
    deblurred_images = []

    with torch.no_grad():
        for blur, sharp, filenames in tqdm(test_loader, desc="Testing"):
            blur, sharp = blur.to(DEVICE), sharp.to(DEVICE)

            # Mengukur waktu inferensi per gambar
            start  = time.time()
            output = model(blur)
            end    = time.time()
            inference_times.append(end - start)

            # Menghitung metrik evaluasi
            test_psnr += compute_psnr(output, sharp)
            test_ssim += compute_ssim(output, sharp)

            # Menyimpan hasil deblurred ke folder Drive
            filename    = filenames[0]
            output_img  = denormalize(output)
            out_path    = os.path.join(OUTPUT_DIR, filename)
            cv2.imwrite(out_path, output_img)

            # Menyimpan ke dalam list untuk dikembalikan jika return_images=True
            if return_images:
                output_img_rgb = cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB)
                deblurred_images.append((filename, output_img_rgb))

    test_psnr /= len(test_loader)
    test_ssim /= len(test_loader)
    avg_inference = np.mean(inference_times) * 1000

    print(f"\nTest PSNR            : {test_psnr:.4f} dB")
    print(f"Test SSIM            : {test_ssim:.4f}")
    print(f"Avg Inference Time   : {avg_inference:.2f} ms/image")
    print(f"Deblurred outputs    : {OUTPUT_DIR}")

    return test_psnr, test_ssim, avg_inference, deblurred_images

def save_results_summary(total_training_time, total_params,
                          trainable_params, flops,
                          test_psnr, test_ssim, avg_inference):
    summary_path = os.path.join(RESULTS_DIR, "results_summary.txt")
    with open(summary_path, "w") as f:
        f.write("HYBRID ViT U-NET DEBLURRING — RESULTS SUMMARY\n")
        f.write("="*50 + "\n\n")
        f.write("COMPUTATIONAL METRICS\n")
        f.write(f"  Total parameters     : {total_params:,}  ({total_params/1e6:.2f}M)\n")
        f.write(f"  Trainable parameters : {trainable_params:,}\n")
        f.write(f"  FLOPs                : {flops/1e9:.2f}G\n" if flops else "  FLOPs : N/A\n")
        f.write(f"  Total training time  : {total_training_time/60:.2f} minutes\n")
        f.write(f"  Avg inference time   : {avg_inference:.2f} ms/image\n\n")
        f.write("IMAGE QUALITY METRICS (test set)\n")
        f.write(f"  Test PSNR            : {test_psnr:.4f} dB\n")
        f.write(f"  Test SSIM            : {test_ssim:.4f}\n\n")
        f.write("TRAINING CONFIG\n")
        f.write(f"  Epochs               : {EPOCHS} (with early stopping, patience={PATIENCE})\n")
        f.write(f"  Batch size           : {BATCH_SIZE}\n")
        f.write(f"  Learning rate        : {LR}\n")
        f.write(f"  Optimizer            : AdamW\n")
        f.write(f"  Loss function        : L1 + SSIM\n")
        f.write(f"  Image size           : {IMG_SIZE}x{IMG_SIZE}\n")
        f.write(f"  Architecture         : Hybrid ViT U-Net\n")
        f.write(f"  Base Features        : [64, 128, 256, 512]\n")
        f.write(f"  Transformer Blocks   : {4} blocks di bottleneck\n")

    print(f"\nResults summary saved: {summary_path}")

In [ ]:
model = ViTUNetDeblur(
    img_size=IMG_SIZE,
    features=[64, 128, 256, 512],
    embed_dim=512,
    depth=4,
    num_heads=8
).to(DEVICE)

criterion = L1SSIMLoss(alpha=0.5)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# 1. Jalankan Training
total_training_time, total_params, trainable_params, flops = train_model(
    model, optimizer, criterion, scheduler)

# 2. Jalankan Testing (Sekarang menangkap list hasil_gambar)
test_psnr, test_ssim, avg_inference, hasil_gambar = test_model(model, return_images=True)

# 3. Simpan Ringkasan Metrik Akhir
save_results_summary(total_training_time, total_params, trainable_params,
                     flops, test_psnr, test_ssim, avg_inference)

COMPUTATIONAL METRICS
Total parameters     : 30,407,491
Trainable parameters : 30,407,491 (30.41M)
FLOPs                : 54.55G



Epoch 001 | Train Loss: 0.2774 | Val Loss: 0.1987 | Val PSNR: 20.17dB | Val SSIM: 0.7201 | Time: 484.8s
  --> Best model saved (val loss: 0.1987)


Epoch 002 | Train Loss: 0.2018 | Val Loss: 0.1638 | Val PSNR: 23.11dB | Val SSIM: 0.7541 | Time: 491.7s
  --> Best model saved (val loss: 0.1638)


Epoch 003 | Train Loss: 0.1741 | Val Loss: 0.1348 | Val PSNR: 24.85dB | Val SSIM: 0.7953 | Time: 492.4s
  --> Best model saved (val loss: 0.1348)


Epoch 004 | Train Loss: 0.1571 | Val Loss: 0.1208 | Val PSNR: 25.73dB | Val SSIM: 0.8180 | Time: 492.8s
  --> Best model saved (val loss: 0.1208)


Epoch 005 | Train Loss: 0.1429 | Val Loss: 0.1148 | Val PSNR: 26.11dB | Val SSIM: 0.8285 | Time: 492.7s
  --> Best model saved (val loss: 0.1148)


... (23 outputs left out)

Epoch 029 | Train Loss: 0.0862 | Val Loss: 0.0792 | Val PSNR: 29.61dB | Val SSIM: 0.8808 | Time: 492.4s


Epoch 030 | Train Loss: 0.0852 | Val Loss: 0.0805 | Val PSNR: 29.61dB | Val SSIM: 0.8789 | Time: 492.7s

TESTING
Loaded best model from: /content/drive/MyDrive/Vit_U_Net_Result/best_model.pth


Testing: 100%|██████████| 624/624 [00:22<00:00, 27.51it/s]



Test PSNR            : 29.6120 dB
Test SSIM            : 0.8822
Avg Inference Time   : 12.11 ms/image
Deblurred outputs    : /content/drive/MyDrive/Vit_U_Net_Result/deblurred_outputs

Results summary saved: /content/drive/MyDrive/Vit_U_Net_Result/results_summary.txt
